# Parallelismo in Python

Esploreremo i principali strumenti per il calcolo parallelo in Python:

## Perché utilizzare il parallelismo?

- Sfruttare tutte le risorse hardware disponibili
- Velocizzare operazioni CPU-bound o I/O-bound
- Gestire più attività simultaneamente

### I/O-bound vs CPU-bound

- **I/O-bound**: operazioni il cui collo di bottiglia è l'attesa di input/output, ad esempio leggere/scrivere file su disco, effettuare richieste di rete, interrogare un database. Il programma passa la maggior parte del tempo *in attesa* che il dato arrivi, non a calcolare.
- **CPU-bound**: operazioni il cui collo di bottiglia è la potenza di calcolo, ad esempio calcoli numerici, compressione dati, rendering, training di modelli ML. Il programma passa la maggior parte del tempo *a calcolare*, non ad aspettare.

Questa distinzione è fondamentale perché determina quale strategia di parallelismo è efficace.

Esistono due approcci principali:
- **Multithreading**: condivisione memoria, ideale per operazioni I/O-bound (i thread possono aspettare contemporaneamente, anche con il GIL)
- **Multiprocessing**: processi separati, ideale per operazioni CPU-bound (ogni processo ha il suo interprete, bypassando il GIL)

## Il problema del GIL (Global Interpreter Lock)

- Il GIL impedisce l'esecuzione parallela di thread Python sulla stessa CPU
- Un solo thread può eseguire codice Python alla volta
- Limita il parallelismo reale nel multithreading
- Il multiprocessing bypassa questo limite creando interpreti Python separati

## Funzione di test

Per confrontare i vari metodi, useremo una funzione che calcola i numeri primi fino a un certo limite

In [1]:
def is_prime(n):
    """Verifica se un numero è primo."""
    if n <= 1:
        return False
    elif n <= 3:
        return True
    elif n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

In [2]:
def count_primes(start, end):
    """Conta i numeri primi in un intervallo"""
    count = 0
    for n in range(start, end):
        if is_prime(n):
            count += 1
    return count

def sequential_primes(ranges):
    """Esecuzione sequenziale"""
    results = []
    for start, end in ranges:
        results.append(count_primes(start, end))
    return results

## Approccio Sequenziale (baseline)

In [3]:
# Definiamo gli intervalli da calcolare
ranges = [(i*1000000, (i+1)*1000000) for i in range(3)]

In [4]:
%%time
sequential_primes(ranges)

CPU times: user 6.21 s, sys: 13 ms, total: 6.22 s
Wall time: 7.86 s


[78498, 70435, 67883]

# 1. Multithreading

## Multithreading con `threading`

- Libreria standard di Python
- Crea thread condividendo lo stesso spazio di memoria
- Soggetto alle limitazioni del GIL
- Ideale per task I/O-bound (networking, file I/O)
- Non ottimale per task CPU-bound (calcoli)

In [5]:
import threading

def threading_primes(ranges):
    results = [0] * len(ranges)

    def worker(idx, start, end):
        results[idx] = count_primes(start, end)

    threads = []
    for i, (start, end) in enumerate(ranges):
        thread = threading.Thread(target=worker, args=(i, start, end))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    return results

In [6]:
%%time
threading_primes(ranges)

CPU times: user 5.08 s, sys: 28.1 ms, total: 5.11 s
Wall time: 5.11 s


[78498, 70435, 67883]

# 2. Multiprocessing

## Multiprocessing con `multiprocessing`

- Crea processi Python separati
- Ogni processo ha il suo interprete Python e memoria
- Bypassa il GIL
- Ideale per task CPU-bound
- Overhead maggiore per la creazione dei processi

In [7]:
import multiprocessing as mp

def multiprocessing_primes(ranges):
    with mp.Pool() as pool:
        # Uso starmap per passare gli argomenti espansi
        results = pool.starmap(count_primes, ranges)
    return results

Multiprocessing non è supportato negli Jupyter Notebook, quindi eseguiremo un file esterno con lo stesso codice tramite subprocess

In [8]:
%%time
import subprocess

process = subprocess.run(
    ["python", "test_multiprocessing.py", "--method", "multiprocessing"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
process.stdout.decode().strip()

CPU times: user 1.92 ms, sys: 10 µs, total: 1.93 ms
Wall time: 33.1 ms


''

---

# 📖 Approfondimento Autonomo

Le sezioni seguenti coprono API alternative per threading e multiprocessing (`concurrent.futures`) e la libreria Joblib, molto usata nell'ecosistema scikit-learn.

## Concurrent.futures con ThreadPoolExecutor

- API più moderna e semplice
- Gestisce automaticamente il pool di thread
- Offre pattern come map/submit per esecuzioni parallele
- Sempre con le limitazioni del GIL

In [9]:
from concurrent.futures import ThreadPoolExecutor

def concurrent_threading_primes(ranges):
    with ThreadPoolExecutor() as executor:
        # Uso map per applicare la funzione a tutti gli elementi
        results = list(executor.map(lambda r: count_primes(r[0], r[1]), ranges))
    return results

In [10]:
%%time
concurrent_threading_primes(ranges)

CPU times: user 5.55 s, sys: 22.1 ms, total: 5.57 s
Wall time: 5.65 s


[78498, 70435, 67883]

## Concurrent.futures con ProcessPoolExecutor

- Stessa API di ThreadPoolExecutor ma usa processi
- Interfaccia più moderna e semplice
- Sotto il cofano usa multiprocessing

In [11]:
from concurrent.futures import ProcessPoolExecutor

def concurrent_processing_primes(ranges):
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(lambda r: count_primes(r[0], r[1]), ranges))
    return results

Anche ProcessPoolExecutor non è supportato negli Jupyter Notebook, quindi eseguiremo un file esterno con lo stesso codice tramite subprocess

In [12]:
%%time
import subprocess

process = subprocess.run(
    ["python", "test_multiprocessing.py", "--method", "concurrent"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
process.stdout.decode().strip()

CPU times: user 1.33 ms, sys: 3 µs, total: 1.33 ms
Wall time: 26.7 ms


''

## Joblib

- Libreria popolare nell'ecosistema scientifico (scikit-learn)
- Supporta sia multiprocessing che multithreading
- Offre caching e pipeline di calcolo
- Gestisce automaticamente la parallelizzazione
- Backend intercambiabili (loky, multiprocessing, threading, dask)

In [13]:
from joblib import Parallel, delayed

def joblib_primes(ranges):
    # n_jobs=-1 utilizza tutti i core disponibili
    results = Parallel(n_jobs=-1)(delayed(count_primes)(start, end) for start, end in ranges)
    return results

In [14]:
%%time
joblib_primes(ranges)

CPU times: user 27.3 ms, sys: 5.07 ms, total: 32.3 ms
Wall time: 4.87 s


[78498, 70435, 67883]